In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
# Imports
import pandas as pd
import numpy as np
from datetime import datetime

In [3]:
# Checking preprocess.py
import sys
sys.path.append("../src")
from data.preprocess import load_ratings

df = load_ratings("../data/raw/ml-1m/ratings.dat")
df.head()

,user_id,movie_id,rating,timestamp
0,1,1193,5,978300760
1,1,661,3,978302109
2,1,914,3,978301968
3,1,3408,4,978300275
4,1,2355,5,978824291


In [4]:
df.shape

(1000209, 4)

In [5]:
# Filtering positives as per threshold (i.e. >=4)
from data.preprocess import filter_positive

positives = filter_positive(df)
positives.shape

(575281, 4)

In [6]:
# Test train split
from data.preprocess import temporal_split
train, test = temporal_split(positives)
train.shape, test.shape

((569243, 4), (6038, 4))

In [7]:
# Number of users who have not rated any movie positively as per set threshold
all_users = set(df["user_id"].unique())
remaining_users = set(positives["user_id"].unique())
missing_users = all_users - remaining_users
print(missing_users)

{np.int64(4486), np.int64(3598)}


In [8]:
# Cross verification
df[df["user_id"] == 3598]["rating"].value_counts()

rating
1    64
2     1
Name: count, dtype: int64

In [9]:
df[df["user_id"] == 4486]["rating"].value_counts()

rating
1    49
2     1
3     1
Name: count, dtype: int64

In [10]:
# Number of movies with no positive ratings as per set threshold
all_movies = set(df["movie_id"].unique())
remaining_movies = set(positives["movie_id"].unique())
missing_movies = all_movies - remaining_movies
len(missing_movies)

173

In [11]:
# Converting user and movie ids to continuos numbers
from data.preprocess import build_id_mappings
user_to_idx, movie_to_idx = build_id_mappings(positives)
len(user_to_idx), len(movie_to_idx)

(6038, 3533)

In [12]:
# Apply mappings
from data.preprocess import apply_mappings
train_mapped = apply_mappings(train, user_to_idx, movie_to_idx)
test_mapped = apply_mappings(test, user_to_idx, movie_to_idx)
train_mapped.head()

,user_id,movie_id,rating,timestamp
31,0,23,4,978300019
22,0,16,5,978300055
27,0,20,4,978300055
37,0,29,5,978300055
36,0,28,5,978300172


In [13]:
train_mapped["user_id"].min(), train_mapped["user_id"].max()

(np.int64(0), np.int64(6037))

In [14]:
train_mapped["movie_id"].min(), train_mapped["movie_id"].max()

(np.int64(0), np.int64(3532))

In [15]:
# Save the processed data
from data.preprocess import save_processed
save_processed(train_mapped, test_mapped, user_to_idx, movie_to_idx, threshold=4)

In [16]:
check = pd.read_csv("../data/processed/train.csv")
check.shape
check.head()

,user_id,movie_id,rating,timestamp
0,0,23,4,978300019
1,0,16,5,978300055
2,0,20,4,978300055
3,0,29,5,978300055
4,0,28,5,978300172


In [17]:
import os
os.listdir("../data/processed")

['metadata.json',
 'movie_to_idx.pkl',
 'test.csv',
 'train.csv',
 'user_to_idx.pkl']

In [18]:
import json
with open("../data/processed/metadata.json") as f:
    print(json.load(f))

{'threshold': 4, 'num_users': 6038, 'num_movies': 3533}
